In [34]:
import numpy as np
import pandas as pd
import plotly.express as px
import plotly.graph_objects as go
import plotly.figure_factory as ff

from plotly.subplots import make_subplots
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

df = pd.read_csv("files/advertising.csv")

In [35]:
df.head(15)

,Daily Time Spent on Site,Age,Area Income,Daily Internet Usage,Ad Topic Line,City,Male,Country,Timestamp,Clicked on Ad
0,68.95,35,61833.90,256.09,Cloned 5thgeneration orchestration,Wrightburgh,0,Tunisia,2016-03-27 00:53:11,0
1,80.23,31,68441.85,193.77,Monitored national standardization,West Jodi,1,Nauru,2016-04-04 01:39:02,0
2,69.47,26,59785.94,236.50,Organic bottom-line service-desk,Davidton,0,San Marino,2016-03-13 20:35:42,0
3,74.15,29,54806.18,245.89,Triple-buffered reciprocal time-frame,West Terrifurt,1,Italy,2016-01-10 02:31:19,0
4,68.37,35,73889.99,225.58,Robust logistical utilization,South Manuel,0,Iceland,2016-06-03 03:36:18,0
5,59.99,23,59761.56,226.74,Sharable client-driven software,Jamieberg,1,Norway,2016-05-19 14:30:17,0
6,88.91,33,53852.85,208.36,Enhanced dedicated support,Brandonstad,0,Myanmar,2016-01-28 20:59:32,0
7,66.00,48,24593.33,131.76,Reactive local challenge,Port Jefferybury,1,Australia,2016-03-07 01:40:15,1
8,74.53,30,68862.00,221.51,Configurable coherent function,West Colin,1,Grenada,2016-04-18 09:33:42,0
9,69.88,20,55642.32,183.82,Mandatory homogeneous architecture,Ramirezton,1,Ghana,2016-07-11 01:42:51,0


In [36]:
df.describe()

,Daily Time Spent on Site,Age,Area Income,Daily Internet Usage,Male,Clicked on Ad
count,1000.000000,1000.000000,1000.000000,1000.000000,1000.000000,1000.00000
mean,65.000200,36.009000,55000.000080,180.000100,0.481000,0.50000
std,15.853615,8.785562,13414.634022,43.902339,0.499889,0.50025
min,32.600000,19.000000,13996.500000,104.780000,0.000000,0.00000
25%,51.360000,29.000000,47031.802500,138.830000,0.000000,0.00000
50%,68.215000,35.000000,57012.300000,183.130000,0.000000,0.50000
75%,78.547500,42.000000,65470.635000,218.792500,1.000000,1.00000
max,91.430000,61.000000,79484.800000,269.960000,1.000000,1.00000


In [37]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1000 entries, 0 to 999
Data columns (total 10 columns):
 #   Column                    Non-Null Count  Dtype  
---  ------                    --------------  -----  
 0   Daily Time Spent on Site  1000 non-null   float64
 1   Age                       1000 non-null   int64  
 2   Area Income               1000 non-null   float64
 3   Daily Internet Usage      1000 non-null   float64
 4   Ad Topic Line             1000 non-null   object 
 5   City                      1000 non-null   object 
 6   Male                      1000 non-null   int64  
 7   Country                   1000 non-null   object 
 8   Timestamp                 1000 non-null   object 
 9   Clicked on Ad             1000 non-null   int64  
dtypes: float64(3), int64(3), object(4)
memory usage: 78.2+ KB


In [38]:
df.columns

Index(['Daily Time Spent on Site', 'Age', 'Area Income',
       'Daily Internet Usage', 'Ad Topic Line', 'City', 'Male', 'Country',
       'Timestamp', 'Clicked on Ad'],
      dtype='object')

In [39]:
for column in df.columns:
    print(f"{column}: {len(df[column].unique())} unique")

Daily Time Spent on Site: 900 unique
Age: 43 unique
Area Income: 1000 unique
Daily Internet Usage: 966 unique
Ad Topic Line: 1000 unique
City: 969 unique
Male: 2 unique
Country: 237 unique
Timestamp: 1000 unique
Clicked on Ad: 2 unique


In [40]:
male_count = df[df["Male"] == 1].shape[0]
print(f"Male count {male_count}")

Male count 481


In [41]:
df['Timestamp'] = pd.to_datetime(df['Timestamp'])
df['Hour'] = df['Timestamp'].dt.hour

clicks_by_hour = df.groupby(['Hour', 'Clicked on Ad']).size().reset_index(name='Count')

fig = px.bar(clicks_by_hour, x='Hour', y='Count', color='Clicked on Ad', barmode='group')

fig.update_layout(width=800, height=400)
fig.update_layout(title='Distribution of clicks by time of day')
fig.show()

In [42]:
fig = px.histogram(df, x="Age", color="Clicked on Ad",
                   marginal="box",
                   hover_data=df.columns)

fig.update_layout(width=800, height=400)
fig.update_layout(title='Histogram of Age and Clicked on Ad')
fig.show()

In [43]:
fig = px.scatter(df, x='Age', y='Area Income', marginal_x='histogram', marginal_y='histogram')


fig.update_layout(width=800, height=400)
fig.update_layout(title='Jointplot of Age and Area Income')
fig.show()

In [44]:
not_object_columns = df.select_dtypes(include=['int', 'float']).columns

fig = make_subplots(rows=2, cols=4, subplot_titles=[f"Distribution of {feature}" for feature in not_object_columns])

for i, feature in enumerate(not_object_columns, 1):
    row = (i - 1) // 4 + 1
    col = (i - 1) % 4 + 1
    
    fig.add_trace(go.Histogram(x=df[feature], marker_color='blue', opacity=0.7, name=feature), row=row, col=col)
    
    mean_value = np.mean(df[feature])
    fig.add_trace(go.Scatter(x=[mean_value, mean_value], y=[0, 15], mode='lines', name='Mean', 
                             line=dict(color='red', width=5)), row=row, col=col)
    
    median = np.median(df[feature])
    fig.add_trace(go.Scatter(x=[median, median], y=[0, 15], mode='lines', name='Median', 
                             line=dict(color='green', width=5)), row=row, col=col)
    
    fig.update_xaxes(title_text=feature, row=row, col=col)
    fig.update_yaxes(title_text="Count", row=row, col=col)

fig.update_layout(height=((len(not_object_columns) - 1) // 4 + 1) * 400, width=1200, 
                  title_text="Distribution of Numeric Features")
fig.show()

In [45]:
corr_matrix = df.corr(numeric_only=True)

fig = px.imshow(corr_matrix,
                width=1500,
                height=1500,
                text_auto=True, aspect="auto",
                color_continuous_scale='RdBu_r',
                )
fig.update_layout(width=800, height=600)
fig.update_layout(title='Correlation Matrix')
fig.show()

In [46]:
corr_matrix_filtered = corr_matrix.mask(abs(corr_matrix) <= 0.25)

corr_clicked_on_ad = corr_matrix_filtered[['Clicked on Ad']].dropna()

fig = px.imshow(corr_clicked_on_ad,
                text_auto=True,
                aspect="auto",
                color_continuous_scale='RdBu_r',
                width=800,
                height=600)

fig.update_layout(width=800, height=600)
fig.update_layout(title='Correlation Matrix with Clicked on Ad')
fig.show()

Не используя sklearn

In [47]:
X_df = df[["Daily Time Spent on Site", "Age", "Area Income", "Daily Internet Usage"]]
y_df = df[["Clicked on Ad"]]

X_train, X_test, y_train, y_test = train_test_split(X_df, y_df, 
                                                    test_size=0.2, random_state=42)

scaler = StandardScaler().fit(X_train)
X_train_scaled = scaler.transform(X_train)
X_test_scaled = scaler.transform(X_test)

In [48]:
def logistic_regression(X, y, learning_rate=0.01, epochs=1000):
    n = len(X)
    w0 = 0
    w1 = np.zeros(X.shape[1])
    
    y = y.reshape(-1)
    
    for epoch in range(epochs):
        linear_model = w0 + np.dot(X, w1)
        y_pred = 1 / (1 + np.exp(-linear_model))
        
        dw0 = np.sum(y_pred - y) / n
        dw1 = np.dot(X.T, (y_pred - y)) / n
        
        w0 -= learning_rate * dw0
        w1 -= learning_rate * dw1
    
    return w0, w1

In [49]:
w0, w1 = logistic_regression(X_train_scaled, y_train.values.ravel())
print(f"Weights {w0}, {w1}")

Weights 0.046031914287381175, [-1.11762868  0.60604021 -0.66170014 -1.27769438]


In [50]:
y_pred = w0 + np.dot(X_test_scaled, w1)
y_pred = np.array([1 if i > 0.5 else 0 for i in y_pred])

In [51]:
def precision(y_true, y_pred):
    tp = np.sum((y_true == 1) & (y_pred == 1))
    fp = np.sum((y_true == 0) & (y_pred == 1))
    return tp / (tp + fp)

def recall(y_true, y_pred):
    tp = np.sum((y_true == 1) & (y_pred == 1))
    fn = np.sum((y_true == 1) & (y_pred == 0))
    return tp / (tp + fn)

def f1(y_true, y_pred):
    prec = precision(y_true, y_pred)
    rec = recall(y_true, y_pred)
    return 2 * (prec * rec) / (prec + rec)

def acc(y_true, y_pred):
    return np.sum(y_true == y_pred)/len(y_true)

In [52]:
y_test = y_test.values.ravel()

print(f"Accuracy: {acc(y_test, y_pred)}")
print(f"Precision: {precision(y_test, y_pred)}")
print(f"Recall: {f1(y_test, y_pred)}")
print(f"F1: {acc(y_test, y_pred)}")

Accuracy: 0.915
Precision: 0.9895833333333334
Recall: 0.9178743961352658
F1: 0.915


Используя sklearn

In [53]:
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score

model = LogisticRegression()
model.fit(X_train, y_train)

y_pred = model.predict(X_test)

print(f"Accuracy: {accuracy_score(y_test, y_pred)}")
print(f"Precision: {precision_score(y_test, y_pred)}")
print(f"Recall: {recall_score(y_test, y_pred)}")
print(f"F1: {f1_score(y_test, y_pred)}")

Accuracy: 0.9
Precision: 0.9595959595959596
Recall: 0.8558558558558559
F1: 0.9047619047619048


c:\Users\user\AppData\Local\Programs\Python\Python310\lib\site-packages\sklearn\utils\validation.py:1183: DataConversionWarning:

A column-vector y was passed when a 1d array was expected. Please change the shape of y to (n_samples, ), for example using ravel().

